# Generate DWS-Bench thesis figures

Generates the six core thesis figures from the prediction artifacts written by the `run_*.ipynb` notebooks
(`<model>/<cot|no_cot>_<tokens>/full_benchmark_predictions.jsonl`).

1. Overall model x prompting accuracy
2. RQ1 accuracy by temporal depth (`rq1_mutation_depth`)
3. RQ2 revision accuracy heatmap (`rq2_supersession`, `revision` family)
4. RQ3 distractor accuracy heatmap (`rq2_interference`, `interleaved_chain`, N = 0)
5. Final-answer versus step-wise accuracy
6. Structural-operation family accuracy heatmap (`rq1_mutation_depth`)

**On Kaggle:** add the result ZIPs/folders of the six Qwen run notebooks (`dws_qwen*_cot`, `dws_qwen*_no_cot`) or
`dws_controlled_reruns` as inputs (or run in the same session). Predictions are discovered automatically under
`DWS_RESULTS_ROOT`, `/kaggle/working`, `/kaggle/input` (zips are unpacked) and `<repo>/results`; `*_MOCK` folders are ignored.
Figures are written to `/kaggle/working/thesis_figures` (and zipped). Locally: `<repo>/report/images/thesis_figures`.


In [ ]:
from pathlib import Path
import json, os, re, shutil, zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / 'run_eval.py').exists():
            return candidate
    return None


ON_KAGGLE = Path('/kaggle').exists()
REPO_ROOT = find_repo_root(Path.cwd().resolve())
WORK_ROOT = Path(os.environ.get('DWS_WORK_ROOT', '/kaggle/working' if ON_KAGGLE else '.')).resolve()
SEARCH_ROOTS = []
if os.environ.get('DWS_RESULTS_ROOT'):
    SEARCH_ROOTS.append(Path(os.environ['DWS_RESULTS_ROOT']))
SEARCH_ROOTS += [WORK_ROOT, Path(os.environ.get('DWS_INPUT_ROOT', '/kaggle/input'))]
if REPO_ROOT:
    SEARCH_ROOTS.append(REPO_ROOT / 'results')
UNZIP_ROOT = WORK_ROOT / '_unzipped_results'

if os.environ.get('DWS_FIGURE_DIR'):
    OUTPUT_DIR = Path(os.environ['DWS_FIGURE_DIR'])
elif ON_KAGGLE or REPO_ROOT is None:
    OUTPUT_DIR = WORK_ROOT / 'thesis_figures'
else:
    OUTPUT_DIR = REPO_ROOT / 'report' / 'images' / 'thesis_figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MAKE_FACTOR_CORRELATION = False
DPI = 300

# Experiment tags come from the generator: the revision (supersession) sweep is tagged
# rq2_supersession, the distractor sweep is the interleaved_chain family of rq2_interference.
RQ1_EXPERIMENT = 'rq1_mutation_depth'
RQ2_EXPERIMENT = 'rq2_interference'
SUPERSESSION_EXPERIMENT = 'rq2_supersession'
DISTRACTOR_FAMILY = 'interleaved_chain'
REVISION_FAMILY = 'revision'
PREFERRED_TOKEN_BUDGET = 256

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 10, 'axes.titlesize': 13, 'axes.labelsize': 11,
    'xtick.labelsize': 9, 'ytick.labelsize': 9, 'figure.dpi': 120,
})

# (size label, prompting label, registry model key, mode directory prefix)
CONFIGS = [
    ('0.5B', 'No CoT', 'qwen2.5-0.5b', 'no_cot'),
    ('0.5B', 'CoT', 'qwen2.5-0.5b', 'cot'),
    ('3B', 'No CoT', 'qwen2.5-3b', 'no_cot'),
    ('3B', 'CoT', 'qwen2.5-3b', 'cot'),
    ('7B', 'No CoT', 'qwen2.5-7b', 'no_cot'),
    ('7B', 'CoT', 'qwen2.5-7b', 'cot'),
]
CONFIG_LABELS = [f'Qwen2.5-{size} {prompt}' for size, prompt, _, _ in CONFIGS]
COLORS = {'No CoT': '#0F766E', 'CoT': '#E07A5F'}


In [ ]:
PRED_NAME = 'full_benchmark_predictions.jsonl'
LAYOUT = re.compile(r'^(?P<mode>cot|no_cot)_(?P<budget>\d+)$')

# Kaggle does not unpack zipped notebook outputs: extract dws_*.zip archives once.
for root in SEARCH_ROOTS:
    if root.exists():
        for z in sorted(root.rglob('dws_*.zip')):
            if '_MOCK' in z.name or UNZIP_ROOT in z.parents:
                continue
            dest = UNZIP_ROOT / z.stem
            if not dest.exists():
                dest.mkdir(parents=True)
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(dest)
SEARCH_ROOTS.append(UNZIP_ROOT)

FOUND = {}   # (model, mode, budget) -> path ; first search root wins
for root in SEARCH_ROOTS:
    if not root.exists():
        continue
    for path in sorted(root.rglob(PRED_NAME)):
        if '_MOCK' in str(path):
            continue
        match = LAYOUT.match(path.parent.name)
        if match:
            FOUND.setdefault((path.parent.parent.name, match['mode'], int(match['budget'])), path)
print('Prediction files found:', len(FOUND))


def choose_prediction_file(model_key, mode):
    options = {b: p for (m, md, b), p in FOUND.items() if m == model_key and md == mode}
    if not options:
        raise FileNotFoundError(f'No {PRED_NAME} for {model_key} / {mode}. Run the matching run_*.ipynb first '
                                f'or point DWS_RESULTS_ROOT at the results.')
    if PREFERRED_TOKEN_BUDGET in options:
        budget = PREFERRED_TOKEN_BUDGET
    else:
        budget = min(options, key=lambda b: (abs(b - PREFERRED_TOKEN_BUDGET), b))
        print(f'WARNING: {model_key} {mode}: no {PREFERRED_TOKEN_BUDGET}-token run, using {budget} tokens.')
    return budget, options[budget]


def load_prediction_file(path, model_size, prompting, budget):
    rows = []
    with path.open(encoding='utf-8') as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            row['model_size'] = model_size
            row['prompting'] = prompting
            row['budget'] = budget
            row['config_label'] = f'Qwen2.5-{model_size} {prompting}'
            row['requested_factors'] = row.get('requested_factors') or {}
            row['measured_factors'] = row.get('measured_factors') or {}
            rows.append(row)
    return rows


all_rows = []
selected_files = {}
for model_size, prompting, model_key, mode in CONFIGS:
    budget, path = choose_prediction_file(model_key, mode)
    selected_files[(model_size, prompting)] = (budget, path)
    all_rows.extend(load_prediction_file(path, model_size, prompting, budget))

predictions = pd.DataFrame(all_rows)
predictions['is_correct'] = predictions['is_correct'].astype(bool)   # strict = semantic AND protocol-compliant
if 'semantic_correct' in predictions:
    predictions['semantic_correct'] = predictions['semantic_correct'].astype(bool)
predictions['step_accuracy'] = (pd.to_numeric(predictions['step_accuracy'], errors='coerce')
                                if 'step_accuracy' in predictions else np.nan)
predictions['experiment'] = predictions['experiment'].fillna('')
predictions['family'] = predictions['family'].fillna('')


def factor(frame, name):
    """Requested factor, falling back to its measured counterpart (E/T/D/V/N carry an _actual suffix)."""
    requested = pd.to_numeric(frame['requested_factors'].map(lambda v: v.get(name, np.nan)))
    key = 'L_word' if name == 'L_word' else f'{name}_actual'
    measured = pd.to_numeric(frame['measured_factors'].map(lambda v: v.get(key, np.nan)))
    return requested.where(requested.notna(), measured)


for name in ['E', 'T', 'D', 'V', 'N', 'L_word']:
    predictions[name] = factor(predictions, name)

# All configurations must have been evaluated on the identical dataset.
reference = None
for (size, prompting), (budget, path) in selected_files.items():
    sub = predictions[(predictions['model_size'] == size) & (predictions['prompting'] == prompting)]
    ident = dict(zip(sub['instance_id'], sub.get('trace_hash', pd.Series(index=sub.index, dtype=object))))
    if reference is None:
        reference = ident
    elif ident != reference:
        raise ValueError(f'{size} {prompting} was evaluated on a different dataset (instance ids / trace hashes differ). '
                         'Re-run it on the same data/full_benchmark.jsonl before plotting.')

print('Loaded rows:', len(predictions))
print('Experiments:', sorted(predictions['experiment'].unique()))
print('Families:', sorted(predictions['family'].unique()))
for key, (budget, path) in selected_files.items():
    print(f'{key[0]} {key[1]} ({budget} tokens) -> {path}')


In [ ]:
def save_figure(fig, filename):
    png_path = OUTPUT_DIR / f'{filename}.png'
    pdf_path = OUTPUT_DIR / f'{filename}.pdf'
    fig.savefig(png_path, dpi=DPI, bbox_inches='tight')
    fig.savefig(pdf_path, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f'Wrote {png_path} and {pdf_path}')


def add_bar_labels(ax, bars):
    for bar in bars:
        value = bar.get_height()
        if np.isfinite(value):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                value + 0.015,
                f'{value:.1%}',
                ha='center',
                va='bottom',
                fontsize=8,
            )


def accuracy_by(frame, group_columns):
    grouped = frame.groupby(group_columns, dropna=False)['is_correct'].mean()
    return grouped.reset_index(name='accuracy')

In [ ]:
rq2 = predictions[predictions['experiment'].isin([RQ2_EXPERIMENT, SUPERSESSION_EXPERIMENT])
                  & (predictions['family'] == REVISION_FAMILY)].copy()
assert not rq2.empty, 'No revision-family records under rq2_supersession / rq2_interference'

rq2_summary = accuracy_by(rq2, ['config_label', 'T'])
rq2_table = rq2_summary.pivot(index='config_label', columns='T', values='accuracy').reindex(CONFIG_LABELS)
fig, ax = plt.subplots(figsize=(8.6, 4.8))
image = ax.imshow(rq2_table * 100, cmap='YlGnBu', vmin=0, vmax=100, aspect='auto')
ax.set_xticks(range(len(rq2_table.columns)), [f'T={int(value)}' for value in rq2_table.columns])
ax.set_yticks(range(len(rq2_table.index)), rq2_table.index)
for row_index in range(rq2_table.shape[0]):
    for column_index in range(rq2_table.shape[1]):
        value = rq2_table.iloc[row_index, column_index]
        if pd.notna(value):
            ax.text(column_index, row_index, f'{value * 100:.0f}', ha='center', va='center', fontsize=8)
ax.set_xlabel('Revision-sweep target depth, $T$')
ax.set_title('RQ2: Revision accuracy heatmap', loc='left', weight='bold')
fig.colorbar(image, ax=ax, label='Final-answer accuracy (%)')
fig.tight_layout()
save_figure(fig, 'fig03_rq2_revision_heatmap')


In [ ]:
overall = accuracy_by(predictions, ['model_size', 'prompting'])
sizes = ['0.5B', '3B', '7B']
x = np.arange(len(sizes))
width = 0.34
fig, ax = plt.subplots(figsize=(7.4, 4.8))
for offset, prompting in [(-width / 2, 'No CoT'), (width / 2, 'CoT')]:
    values = [
        overall.query('model_size == @size and prompting == @prompting')['accuracy'].iloc[0]
        for size in sizes
    ]
    bars = ax.bar(x + offset, values, width, label=prompting, color=COLORS[prompting])
    add_bar_labels(ax, bars)
ax.set_xticks(x, sizes)
ax.set_ylim(0, min(1.0, float(overall['accuracy'].max()) + 0.12))
ax.set_xlabel('Qwen2.5 parameter scale')
ax.set_ylabel('Final-answer accuracy')
ax.set_title('Overall accuracy by model scale and prompting', loc='left', weight='bold')
ax.legend(frameon=False, ncols=2)
fig.tight_layout()
save_figure(fig, 'fig01_overall_model_prompting')

In [ ]:
rq1 = predictions.query("experiment == @RQ1_EXPERIMENT").copy()
rq1_summary = accuracy_by(rq1, ['config_label', 'model_size', 'prompting', 'T'])
fig, ax = plt.subplots(figsize=(9.2, 5.4))
for label in CONFIG_LABELS:
    subset = rq1_summary.query('config_label == @label').sort_values('T')
    if subset.empty:
        continue
    prompting = subset['prompting'].iloc[0]
    ax.plot(
        subset['T'],
        subset['accuracy'] * 100,
        marker='o',
        linewidth=2,
        label=label,
        color=COLORS[prompting],
        alpha=0.55 if prompting == 'No CoT' else 1.0,
        linestyle='--' if prompting == 'No CoT' else '-',
    )
# RQ1 basic_chain sweeps T over these levels; others use {4,8,12,16} or {6,8,12,16}.
ax.set_xticks(sorted(value for value in rq1_summary['T'].dropna().unique()))
ax.set_ylim(0, 105)
ax.set_xlabel('Target-relevant update depth, $T$')
ax.set_ylabel('Final-answer accuracy (%)')
ax.set_title('RQ1: Accuracy by temporal depth', loc='left', weight='bold')
ax.legend(frameon=False, ncols=2, fontsize=8)
fig.tight_layout()
save_figure(fig, 'fig02_rq1_temporal_depth')


In [ ]:
# D=4 is also the base of the N sweep (N=4,8,16); restrict to N=0 so every D level is the pure D effect.
rq3 = predictions[(predictions['experiment'] == RQ2_EXPERIMENT)
                  & (predictions['family'] == DISTRACTOR_FAMILY)
                  & (predictions['N'].fillna(0) == 0)].copy()
assert not rq3.empty, 'No interleaved_chain records with N=0'

rq3_summary = accuracy_by(rq3, ['config_label', 'D'])
rq3_table = rq3_summary.pivot(index='config_label', columns='D', values='accuracy').reindex(CONFIG_LABELS)
fig, ax = plt.subplots(figsize=(8.6, 4.8))
image = ax.imshow(rq3_table * 100, cmap='YlOrRd', vmin=0, vmax=100, aspect='auto')
ax.set_xticks(range(len(rq3_table.columns)), [f'D={int(value)}' for value in rq3_table.columns])
ax.set_yticks(range(len(rq3_table.index)), rq3_table.index)
for row_index in range(rq3_table.shape[0]):
    for column_index in range(rq3_table.shape[1]):
        value = rq3_table.iloc[row_index, column_index]
        if pd.notna(value):
            ax.text(column_index, row_index, f'{value * 100:.0f}', ha='center', va='center', fontsize=8)
ax.set_xlabel('State-changing distractor count, $D$')
ax.set_title('RQ3: Distractor accuracy heatmap (N = 0)', loc='left', weight='bold')
fig.colorbar(image, ax=ax, label='Final-answer accuracy (%)')
fig.tight_layout()
save_figure(fig, 'fig04_rq3_distractor_heatmap')


In [ ]:
metric_rows = []
for label in CONFIG_LABELS:
    subset = predictions.query('config_label == @label')
    final_accuracy = subset['is_correct'].mean()
    step_accuracy = subset['step_accuracy'].dropna().mean()
    if np.isfinite(step_accuracy):
        metric_rows.append({'config_label': label, 'metric': 'Final answer', 'accuracy': final_accuracy})
        metric_rows.append({'config_label': label, 'metric': 'Step-wise', 'accuracy': step_accuracy})
metric_table = pd.DataFrame(metric_rows)
assert not metric_table.empty, 'No step-wise accuracies available (CoT runs missing?)'
fig, ax = plt.subplots(figsize=(9.2, 5.2))
labels = metric_table['config_label'].drop_duplicates().tolist()
x = np.arange(len(labels))
width = 0.36
for offset, metric, color in [(-width / 2, 'Final answer', '#0F766E'), (width / 2, 'Step-wise', '#E07A5F')]:
    values = [metric_table.query('config_label == @label and metric == @metric')['accuracy'].iloc[0] for label in labels]
    bars = ax.bar(x + offset, np.array(values) * 100, width, label=metric, color=color)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1, f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x, labels, rotation=25, ha='right')
ax.set_ylim(0, min(100, float(np.nanmax(metric_table['accuracy'])) * 100 + 8))
ax.set_ylabel('Accuracy (%)')
ax.set_title('Final-answer versus canonical step-wise accuracy', loc='left', weight='bold')
ax.legend(frameon=False, ncols=2)
fig.tight_layout()
save_figure(fig, 'fig05_final_vs_stepwise')

In [ ]:
rq5 = predictions.query("experiment == @RQ1_EXPERIMENT").copy()
operation_order = ['split_chain', 'merge_chain', 'swap_chain', 'undo_chain', 'undo_redo_chain']
operation_labels = ['Split', 'Merge', 'Swap', 'Undo', 'Undo-Redo']
rq5_summary = accuracy_by(rq5, ['config_label', 'family'])
rq5_table = rq5_summary.pivot(index='config_label', columns='family', values='accuracy').reindex(index=CONFIG_LABELS, columns=operation_order)
fig, ax = plt.subplots(figsize=(8.8, 4.8))
image = ax.imshow(rq5_table * 100, cmap='PuBuGn', vmin=0, vmax=100, aspect='auto')
ax.set_xticks(range(len(operation_order)), operation_labels)
ax.set_yticks(range(len(rq5_table.index)), rq5_table.index)
for row_index in range(rq5_table.shape[0]):
    for column_index in range(rq5_table.shape[1]):
        value = rq5_table.iloc[row_index, column_index]
        if pd.notna(value):
            ax.text(column_index, row_index, f'{value * 100:.0f}', ha='center', va='center', fontsize=8)
ax.set_xlabel('Structural-operation family')
ax.set_title('Structural-operation accuracy heatmap', loc='left', weight='bold')
fig.colorbar(image, ax=ax, label='Final-answer accuracy (%)')
fig.tight_layout()
save_figure(fig, 'fig06_operation_family_heatmap')


In [ ]:
# Bundle the figures for download.
if ON_KAGGLE:
    print(shutil.make_archive(str(WORK_ROOT / 'thesis_figures'), 'zip', root_dir=OUTPUT_DIR))
